# Chapter 03-08 · Loss, gradients, and how a model is fitted

**Label:** Core  |  **Time:** ~60 minutes  |  **Difficulty:** moderate - no calculus is required, and
the derivative is computed numerically so you can see what it is

**Prerequisites:** 03-07 for `X @ w`, 03-06 for slopes, 03-01 for the idea that a summary minimises an
error.

**Position in the learning path:** module 03, chapter 8 of 8 - the last mathematics chapter. After
this: **module 04**, the workflow, and then real models.

---

## Why this matters

You can now write a prediction: `X @ w`. Everything that remains is choosing `w`.

03-01 already did this once. It searched a grid of candidate values and kept the one with the smallest
total error - and found that the winner was the mean. That approach works perfectly for one number.
For two it needs ten thousand evaluations, and for ten parameters it needs **a hundred billion
billion**, which is not a slow method but an impossible one.

This chapter replaces the search with a procedure that scales: measure which way is downhill, take a
step, repeat. It is the method behind linear regression, logistic regression, every neural network,
and most of the rest of the field. It takes five lines.

It also has two ways of failing that you will meet constantly, and the second one is the reason
03-07's scaling lesson matters even for methods that have nothing to do with distance.

## What you will be able to do

- Write down a loss function and explain what it measures
- Say why grid search stops working, with a number
- Compute a derivative numerically, without doing any calculus
- Implement gradient descent in five lines and check it against the exact answer
- Diagnose a learning rate that is too small or too large from the loss curve
- Explain why unscaled features make fitting hundreds of times slower

## Warm-up: retrieve, do not reread

1. What does `X @ w` compute, and what is its shape when `X` is `(60, 3)`?
2. Why did raw Euclidean distance pick the flat with the wrong number of rooms?
3. Which methods are unaffected by feature scaling?

<br>

*Answers: (1) a weighted sum per row - a prediction for each row; shape `(60,)`. (2) rent had the
largest numbers, so it supplied nearly all of every squared distance. (3) trees, random forests and
gradient boosting, which split one column at a time.*

## The setup, and the thing we are trying to find

Sixty days of rentals against temperature, with noise. We are looking for a slope and an intercept.

Because this data is synthetic we know the answer - it was built with a slope of 15 and an intercept
of 40 - and `np.polyfit` can solve it exactly. Both are kept aside as a scoreboard. **The point of the
chapter is the method, not the answer.**

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# SYNTHETIC: rentals = 40 + 15 x temperature + noise
rng = np.random.default_rng(7)
n_days = 60
temperature = rng.uniform(10, 30, n_days)
rentals = 40 + 15 * temperature + rng.normal(0, 25, n_days)

exact_slope, exact_intercept = np.polyfit(temperature, rentals, 1)
print("the exact answer, for checking later:")
print("  slope     %.4f   (built with 15)" % exact_slope)
print("  intercept %.4f   (built with 40)" % exact_intercept)

## Loss: a single number for "how wrong"

A **loss function** takes a candidate answer and returns one number saying how badly it fits. For a
regression the usual choice is the **mean squared error** - the average of the squared residuals,
which is 03-01's total squared error divided by the number of rows.

Lower is better, and zero is perfect.

In [ ]:
def loss(slope, intercept):
    predictions = intercept + slope * temperature
    return float(np.mean((rentals - predictions) ** 2))


for candidate_slope in [0, 5, 10, 15, 20]:
    print("slope %2d, intercept 40  ->  loss %10.2f" % (candidate_slope, loss(candidate_slope, 40)))

In [ ]:
slopes = np.linspace(0, 30, 200)
curve = [loss(s, 40) for s in slopes]

fig, ax = plt.subplots(figsize=(7.5, 4))
ax.plot(slopes, curve, color="#0072B2", linewidth=2)
ax.axvline(exact_slope, color="#D55E00", linestyle="--")
ax.text(exact_slope + 0.5, max(curve) * 0.7, "the best slope\n%.2f" % exact_slope, color="#D55E00")
ax.set_xlabel("candidate slope")
ax.set_ylabel("loss (mean squared error)")
ax.set_title("The loss is a bowl. Fitting means finding its bottom")
plt.tight_layout()
plt.show()

**Fitting a model means finding the lowest point of that bowl.** Everything else in this chapter is
about how to get there.

## Why the obvious approach stops working

03-01 found the best value by trying many and keeping the winner. That is fine here - the curve above
is 200 evaluations. Now count what it costs as the number of parameters grows, trying 100 values of
each.

In [ ]:
rows = []
for parameters in [1, 2, 3, 5, 10, 20]:
    evaluations = 100.0 ** parameters
    rows.append({"parameters": parameters,
                 "grid evaluations": "%.0e" % evaluations,
                 "time at 1 microsecond each": "%.1e seconds" % (evaluations * 1e-6)})
print(pd.DataFrame(rows).to_string(index=False))
print()
print("the age of the universe is about 4e17 seconds")

Two parameters is ten thousand evaluations - instant. Ten parameters is **1e+20**, which at a
microsecond each is a hundred trillion seconds, or roughly ten thousand times the age of the universe.

And ten parameters is a *small* model. A modest neural network has millions.

**The problem with grid search is not that it is slow. It is that it learns nothing from each
evaluation.** Every point is tried in isolation, and knowing the loss at one place tells it nothing
about where to look next.

## The derivative: which way is downhill

The fix is to use more information from each evaluation - specifically, **the slope of the loss curve
at the point you are standing on**. If the loss goes down as you increase the parameter, increase it.

That slope is the derivative, and you do not need calculus to get it. **Move a tiny step each way and
see how much the loss changed.**

In [ ]:
def derivative(function, at, step=1e-5):
    return (function(at + step) - function(at - step)) / (2 * step)


here_slope, here_intercept = 10.0, 40.0

d_slope = derivative(lambda s: loss(s, here_intercept), here_slope)
d_intercept = derivative(lambda b: loss(here_slope, b), here_intercept)

print("standing at slope %.1f, intercept %.1f" % (here_slope, here_intercept))
print("  loss                      : %.4f" % loss(here_slope, here_intercept))
print("  derivative wrt slope      : %.4f" % d_slope)
print("  derivative wrt intercept  : %.4f" % d_intercept)
print()
print("both are negative, so increasing either one REDUCES the loss - we are to the left of the bottom")

### What the derivative looks like

A derivative is nothing more exotic than **the steepness of the straight line that just touches the
curve** at the point you are standing on. Negative means the curve is falling to the right, so moving
right lowers the loss.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(slopes, curve, color="#0072B2", linewidth=2, label="loss")

touch = np.linspace(here_slope - 6, here_slope + 6, 2)
ax.plot(touch, loss(here_slope, 40) + d_slope * (touch - here_slope),
        color="#D55E00", linewidth=2, label="tangent at slope 10")
ax.plot([here_slope], [loss(here_slope, 40)], "o", color="#D55E00", markersize=8)
ax.annotate("", xy=(here_slope + 4, loss(here_slope, 40) - 200),
            xytext=(here_slope, loss(here_slope, 40)),
            arrowprops=dict(arrowstyle="-|>", color="#009E73", linewidth=2.5))
ax.text(here_slope + 4.3, loss(here_slope, 40) - 1500,
        "downhill\n(the derivative is negative,\nso increase the slope)", color="#009E73")

ax.set_ylim(0, 45000)
ax.set_xlabel("candidate slope")
ax.set_ylabel("loss")
ax.set_title("The derivative is the tangent's steepness: %.0f at slope 10" % d_slope)
ax.legend()
plt.tight_layout()
plt.show()

### The derivative also tells you *how far*, not only which way

Read the sign and you know the direction. Read the size and you learn something else: **the derivative
shrinks to zero as you approach the bottom.** That is what makes the method self-limiting - the steps
get smaller on their own as the fit improves, with nobody having to slow them down.

**Predict before running:** the derivative is -4185 at slope 10. What do you expect its sign to be at
slope 20 - and roughly what size?

In [ ]:
table = []
for candidate in [0, 5, 8, 10, 12, 15, 18, 20, 25]:
    table.append({"slope": candidate,
                  "loss": round(loss(candidate, 40), 2),
                  "derivative": round(derivative(lambda s: loss(s, 40), float(candidate)), 2)})
table = pd.DataFrame(table)
table["change in derivative"] = table["derivative"].diff().round(2)
table["per unit of slope"] = (table["change in derivative"] / table["slope"].diff()).round(2)
print(table.to_string(index=False, na_rep="-"))

Three things in that table are worth more than the plot:

1. **The derivative crosses zero between 12 and 15.** Its sign is a compass: negative means the bottom
   is to your right, positive means you have gone past it.
2. **Its magnitude falls steadily** as the bottom approaches - 12,681 at slope 0, 4,185 at slope 10,
   2,486 at slope 12. Far away the steps are large; near the bottom they shrink to nothing.
3. **The last column is the same number every time - 849.61, give or take a wobble in the final decimal
   from the finite-difference step.** The derivative changes at a constant rate,
   because a squared-error loss is exactly a parabola. That is a special property of this loss, and it
   is worth one more calculation.

If the derivative is -12,681.61 at slope 0 and rises by 849.61 for every unit of slope, then it reaches
zero after `12681.61 / 849.61` units - and that lands you at the bottom **in a single jump, with no
searching at all**.

In [ ]:
start = 0.0
d_here = derivative(lambda s: loss(s, 40), start)
curvature = 2 * np.mean(temperature ** 2)          # the rate at which the derivative changes

jump = start - d_here / curvature
print("derivative at slope 0 : %.4f" % d_here)
print("rate of change        : %.4f  (this is 2 x mean(temperature^2))" % curvature)
print("one jump lands at     : %.4f" % jump)
print("derivative there      : %.6f" % (round(derivative(lambda s: loss(s, 40), jump), 6) + 0.0))

**Zero to six decimal places, from one evaluation.** Dividing by the curvature instead of guessing a
step size is called **Newton's method**, and where it applies it is unbeatable.

So why does the rest of the chapter not do that? Because the curvature of a model with `p` parameters
is a `p x p` matrix that has to be inverted - about `p^3` work, and memory for `p^2` numbers. At
p = 1,000,000 that is impossible, while a gradient stays a list of a million numbers. **Gradient descent
wins not because it is better but because it is the one that still runs.**

Keep the shape of it, though: dividing by the curvature is the *right* step size, so a learning rate is
really a cheap guess at `1 / curvature`. That single sentence explains both failures in this chapter's
failure labs.

### The same numbers, from calculus

For the mean squared error the derivatives can be worked out on paper, and they come to
`2 x mean(residual x feature)` for the slope and `2 x mean(residual)` for the intercept. Worth checking
that the numerical version agrees.

In [ ]:
residuals = (here_intercept + here_slope * temperature) - rentals

print("numerical  : %.4f  and  %.4f" % (d_slope, d_intercept))
print("analytic   : %.4f  and  %.4f" % (2 * np.mean(residuals * temperature), 2 * np.mean(residuals)))

**Identical to four decimals.** The two lines of arithmetic and the calculus agree, which is worth
seeing once: the derivative is not a mysterious object, it is the answer to *"if I nudge this, how
much does that move?"*

In practice libraries use the analytic form - it is exact and far faster, and for deep networks it is
computed automatically by the chain rule, which is what "backpropagation" means. **The numerical
version remains the best way to check that an analytic gradient is right**, and a mismatch between the
two is the standard test when implementing one.

The two derivatives taken together - one per parameter - are called the **gradient**. It is a vector
pointing in the direction the loss increases fastest, so its negative points downhill.

## Gradient descent, in five lines

Start anywhere. Work out which way is downhill. Take a small step that way. Repeat.

The size of the step is controlled by a number called the **learning rate**, and it is the only thing
you have to choose.

Standardise the feature first - 03-07's habit - and the reason will become brutally clear in the
second failure lab.

In [ ]:
mean_temp, sd_temp = temperature.mean(), temperature.std()
scaled_temp = (temperature - mean_temp) / sd_temp


def fit(feature, learning_rate, steps):
    slope, intercept = 0.0, 0.0
    history = []
    for _ in range(steps):
        residual = (intercept + slope * feature) - rentals      # how wrong, per row
        gradient_slope = 2 * np.mean(residual * feature)        # downhill direction, slope
        gradient_intercept = 2 * np.mean(residual)              # downhill direction, intercept
        slope -= learning_rate * gradient_slope                 # step
        intercept -= learning_rate * gradient_intercept
        history.append(np.mean((rentals - (intercept + slope * feature)) ** 2))
    return slope, intercept, np.array(history)


found_slope, found_intercept, history = fit(scaled_temp, learning_rate=0.05, steps=400)

best_slope, best_intercept = np.polyfit(scaled_temp, rentals, 1)
print("gradient descent : slope %.4f  intercept %.4f  loss %.4f"
      % (found_slope, found_intercept, history[-1]))
print("the exact answer : slope %.4f  intercept %.4f  loss %.4f"
      % (best_slope, best_intercept,
         np.mean((rentals - (best_intercept + best_slope * scaled_temp)) ** 2)))
print()
print("converting back to the original units:")
print("  slope     %.4f  per degree   (exact: %.4f)" % (found_slope / sd_temp, exact_slope))
print("  intercept %.4f               (exact: %.4f)"
      % (found_intercept - found_slope * mean_temp / sd_temp, exact_intercept))

**It finds the exact answer**, to four decimal places, from five lines that never mention regression.

That is the whole idea, and it is worth stating plainly because everything later in the course is a
variation on it: **choose a way of measuring wrongness, then walk downhill.** Change the loss and you
change the model - swap squared error for absolute error and you get a different fit (03-01's fork
again); swap it for log-loss and you get logistic regression; keep the loop and stack a hundred layers
in front of it and you get a neural network. The loop does not change.

In [ ]:
fig, (left, right) = plt.subplots(1, 2, figsize=(12, 4))

left.plot(history, color="#0072B2")
left.set_xlabel("step")
left.set_ylabel("loss")
left.set_title("the loss falls and flattens")

right.plot(history[:60], "o-", color="#0072B2", markersize=3)
right.set_xlabel("step")
right.set_ylabel("loss")
right.set_title("the first 60 steps: most of the work is done early")

plt.tight_layout()
plt.show()
print("loss at step 1: %10.2f" % history[0])
print("loss at step 10: %9.2f" % history[9])
print("loss at step 50: %9.2f" % history[49])
print("loss at step 400: %8.2f" % history[-1])

### Watching it happen

Two views of that same run. The first is the one you would draw for a colleague: the fitted line at
five moments, swinging up out of the floor and settling onto the data.

In [ ]:
def path(feature, learning_rate, steps):
    slope, intercept, visited = 0.0, 0.0, [(0.0, 0.0)]
    for _ in range(steps):
        residual = (intercept + slope * feature) - rentals
        slope -= learning_rate * 2 * np.mean(residual * feature)
        intercept -= learning_rate * 2 * np.mean(residual)
        visited.append((slope, intercept))
    return np.array(visited)


walk = path(scaled_temp, 0.05, 400)

fig, ax = plt.subplots(figsize=(8, 4.8))
ax.scatter(temperature, rentals, s=18, color="#999999", zorder=1)
degrees = np.linspace(9, 31, 2)
for step, shade in zip([1, 3, 10, 30, 100], ["#cfe3f3", "#a6cced", "#7db4e6", "#3f8fd2", "#0072B2"]):
    w, b = walk[step]
    ax.plot(degrees, (b - w * mean_temp / sd_temp) + (w / sd_temp) * degrees,
            color=shade, linewidth=2, zorder=2, label="step %d" % step)
ax.set_xlabel("temperature (C)")
ax.set_ylabel("rentals")
ax.set_title("The line swings into place: five snapshots of one descent")
ax.legend(loc="upper left", fontsize=8)
plt.tight_layout()
plt.show()

The second view is the one that explains *why* it works, and it is the picture to keep.

Draw the loss as a map: slope across, intercept up, colour for height. The result is a **bowl seen from
above** - dark in the middle, where the loss is lowest. Gradient descent is a walk on that map, and each
step points straight down the local slope.

In [ ]:
from matplotlib.colors import LogNorm

slope_axis = np.linspace(-5, 115, 150)
intercept_axis = np.linspace(-30, 420, 150)
height = np.array([[np.mean((rentals - (b + w * scaled_temp)) ** 2) for w in slope_axis]
                   for b in intercept_axis])

fig, ax = plt.subplots(figsize=(7.5, 5.5))
bands = np.geomspace(400, 200000, 25)
ax.contourf(*np.meshgrid(slope_axis, intercept_axis), height, levels=bands,
            norm=LogNorm(), cmap="Blues_r")
ax.contour(*np.meshgrid(slope_axis, intercept_axis), height, levels=bands[::3],
           colors="white", linewidths=0.6)
ax.plot(walk[:60, 0], walk[:60, 1], "o-", color="#D55E00", markersize=3.5, linewidth=1.4)
ax.plot([best_slope], [best_intercept], "*", color="#000000", markersize=16)
ax.set_xlabel("slope")
ax.set_ylabel("intercept")
ax.set_title("The loss from above: 60 steps, and the star is the answer")
plt.tight_layout()
plt.show()

Two features of that walk are worth naming, because both come straight from the derivative table in the
first half:

- **The steps are long at the start and short at the end.** Nobody programmed that. The gradient is
  large far from the bottom and small near it, so the same learning rate produces a long stride out on
  the rim and a shuffle in the middle.
- **The path is essentially straight.** The contours are near-circles, so the downhill direction points
  at the star from wherever you stand. Remember this picture - the second failure lab is the same walk
  on a map whose contours are *not* circles, and the difference costs a factor of two hundred.

The steps shrinking on their own is measurable, and on standardised data it follows an exact law.

In [ ]:
distance = np.hypot(walk[:, 0] - best_slope, walk[:, 1] - best_intercept)
print("step   distance to the star   ratio to the previous step")
for step in [1, 2, 3, 10, 20, 40]:
    print("%4d %18.4f %22.4f" % (step, distance[step], distance[step] / distance[step - 1]))
print()
print("every step closes a fixed fraction of the remaining gap: 1 - 2 x learning rate = %.4f" % (1 - 2 * 0.05))

**0.9 exactly, every step.** Gradient descent does not creep in by a fixed amount, it removes a fixed
*percentage* of the remaining error - which is why the loss curve looks the way it does, and why
"how many steps do I need?" has an answer you can compute rather than guess. Closing a factor of a
thousand at 0.9 per step takes `log(1/1000) / log(0.9)` = about 66 steps, whatever the data.

(The clean 0.9 is a consequence of standardising: it makes the average of the squared feature exactly 1,
so the contraction is exactly `1 - 2 x learning rate`. Set the learning rate to 0.5 and the factor is
zero - one step lands on the answer, which is Newton's method arriving by a different door.)

## Failure lab: the learning rate

The step size is the only setting, and both directions of getting it wrong are common enough that you
should be able to recognise each from the loss curve alone.

**Predict before running:** what happens with a learning rate of 1.01?

In [ ]:
rows = []
for rate in [0.001, 0.05, 0.3, 1.01]:
    _, _, curve = fit(scaled_temp, learning_rate=rate, steps=400)
    final = curve[-1]
    if not np.isfinite(final) or final > curve[0]:
        verdict, shown = "diverging", ("%.4g" % final) if np.isfinite(final) else "overflowed"
    elif final < 1.001 * 431.7840:
        verdict, shown = "arrived", "%.4f" % final
    else:
        verdict, shown = "too small - still descending", "%.4f" % final
    rows.append({"learning rate": rate, "loss after 400 steps": shown, "verdict": verdict})
print(pd.DataFrame(rows).to_string(index=False))
print()
print("for reference, the best achievable loss is %.4f"
      % np.mean((rentals - (best_intercept + best_slope * scaled_temp)) ** 2))

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
for rate, colour in [(0.001, "#D55E00"), (0.05, "#0072B2"), (0.3, "#009E73")]:
    _, _, curve = fit(scaled_temp, learning_rate=rate, steps=400)
    ax.plot(curve, color=colour, label="learning rate %.3f" % rate)
ax.set_yscale("log")
ax.set_xlabel("step")
ax.set_ylabel("loss (log scale)")
ax.legend()
ax.set_title("Too small never arrives; the other two do")
plt.tight_layout()
plt.show()

The table says what happened. This says why, and it is the single most useful picture in the chapter:
each arrow is one step, drawn on the bowl itself.

To keep it to one dimension the intercept is held at its best value, so only the slope moves. That
changes which rates are workable - alone, the slope tolerates far more than it does with the intercept
moving too - but the three shapes are exactly the ones you will see in real loss curves.

In [ ]:
def slope_only_loss(candidate):
    return float(np.mean((rentals - (best_intercept + candidate * scaled_temp)) ** 2))


fig, axes = plt.subplots(1, 3, figsize=(13, 4))
labels = ["cautious: many small steps", "well chosen: lands in two",
          "too large: overshoots further each time"]
for ax, rate, label in zip(axes, [0.05, 0.45, 1.05], labels):
    candidate, visited = 0.0, [0.0]
    for _ in range(6):
        residual = (best_intercept + candidate * scaled_temp) - rentals
        candidate -= rate * 2 * np.mean(residual * scaled_temp)
        visited.append(candidate)

    low, high = min(visited + [best_slope]), max(visited + [best_slope])
    margin = 0.15 * (high - low) + 5
    grid = np.linspace(low - margin, high + margin, 300)
    ax.plot(grid, [slope_only_loss(v) for v in grid], color="#bbbbbb", linewidth=1.8)
    for start, end in zip(visited[:-1], visited[1:]):
        ax.annotate("", xy=(end, slope_only_loss(end)), xytext=(start, slope_only_loss(start)),
                    arrowprops=dict(arrowstyle="-|>", color="#0072B2", linewidth=1.6))
    ax.plot(visited, [slope_only_loss(v) for v in visited], "o", color="#0072B2", markersize=5)
    ax.axvline(best_slope, color="#D55E00", linestyle="--")
    ax.set_title("rate %.2f - %s" % (rate, label), fontsize=10)
    ax.set_xlabel("slope")
    ax.set_ylabel("loss")
plt.tight_layout()
plt.show()

The right-hand panel is the one to remember. **Divergence is not the method going wrong - it is the
method working correctly with too large a step.** Each arrow points genuinely downhill; it simply
travels so far that it lands higher up the far wall than it started. The next gradient is therefore
bigger, so the next jump is longer, and the zigzag widens until the numbers overflow.

That also explains why the cure is always the same: make the step shorter. Nothing about the direction
was wrong.

### Reading the three outcomes

- **0.001 - too small.** After 400 steps the loss is **24,582** against a best possible **431.78**. The
  method is working correctly and moving in the right direction; it will simply take tens of thousands
  of steps. On the log-scale plot this is the line still visibly descending at the right-hand edge,
  which is the signature: *a loss curve that has not flattened has not finished.*
- **0.05 and 0.3 - fine.** Both reach 431.7840, the exact optimum. There is usually a wide band of
  workable rates rather than one correct value.
- **1.01 - diverging.** The loss after 400 steps is **9.092e+11**, against 120,244 at the starting
  point of slope 0 and intercept 0. It has not settled anywhere; it is climbing. Each step overshoots
  the bottom of the bowl and lands further up the opposite side, so the next gradient is larger, the
  next step is longer, and the process runs away. Given a few hundred more steps it overflows to
  `inf`.

**The diagnosis is available from the loss curve alone**, which is why plotting it is the first thing
to do when a fit misbehaves:

| What the curve does | What it means | What to do |
|---|---|---|
| Falls and flattens | Converged | Nothing |
| Still falling at the end | Rate too small, or too few steps | Increase the rate, or run longer |
| Rises, oscillates, or becomes `nan` | Rate too large | Decrease it, by a factor of ten |
| Falls then rises | Usually too large, sometimes a bug in the gradient | Decrease it, then check the gradient numerically |

A reliable starting recipe: try 0.1, and if it diverges divide by ten until it does not.

## Failure lab 2: what unscaled features cost

The feature was standardised before any of that. Here is what happens without it - the identical
algorithm, on the identical data, in the original units.

In [ ]:
def steps_to_converge(feature, learning_rate, tolerance=1e-6, limit=200_000):
    slope, intercept, previous = 0.0, 0.0, np.inf
    for step in range(1, limit + 1):          # a diverging run overflows on purpose here
        with np.errstate(over="ignore", invalid="ignore"):
            residual = (intercept + slope * feature) - rentals
            slope -= learning_rate * 2 * np.mean(residual * feature)
            intercept -= learning_rate * 2 * np.mean(residual)
            current = np.mean((rentals - (intercept + slope * feature)) ** 2)
        if not np.isfinite(current):
            return None, step
        if abs(previous - current) < tolerance:
            return current, step
        previous = current
    return current, limit


rows = []
for label, feature, rate in [("standardised", scaled_temp, 0.05),
                             ("raw degrees", temperature, 0.0005),
                             ("raw degrees", temperature, 0.0010),
                             ("raw degrees", temperature, 0.0015),
                             ("raw degrees", temperature, 0.0100)]:
    final, step = steps_to_converge(feature, rate)
    rows.append({"feature": label, "learning rate": rate,
                 "steps to converge": step if final is not None else "diverged at step %d" % step,
                 "final loss": ("%.4f" % final) if final is not None else "-"})
print(pd.DataFrame(rows).to_string(index=False))

The number is stark, but the map is what makes it obvious. Same algorithm, same data, same picture as
before - only the units of the feature change.

In [ ]:
fig, (left, right) = plt.subplots(1, 2, figsize=(12, 5))

left.contourf(*np.meshgrid(slope_axis, intercept_axis), height, levels=bands,
              norm=LogNorm(), cmap="Blues_r")
left.contour(*np.meshgrid(slope_axis, intercept_axis), height, levels=bands[::3],
             colors="white", linewidths=0.6)
left.plot(walk[:60, 0], walk[:60, 1], "o-", color="#D55E00", markersize=3.5, linewidth=1.4)
left.plot([best_slope], [best_intercept], "*", color="#000000", markersize=16)
left.set_title("standardised: circles, and a straight walk")
left.set_xlabel("slope")
left.set_ylabel("intercept")

raw_slopes = np.linspace(15.2, 18.2, 220)
raw_intercepts = np.linspace(-2, 31, 220)
raw_height = np.array([[np.mean((rentals - (b + w * temperature)) ** 2) for w in raw_slopes]
                       for b in raw_intercepts])
raw_bands = np.geomspace(431, 3000, 25)
right.contourf(*np.meshgrid(raw_slopes, raw_intercepts), raw_height, levels=raw_bands,
               norm=LogNorm(), cmap="Oranges_r", extend="max")
right.contour(*np.meshgrid(raw_slopes, raw_intercepts), raw_height, levels=raw_bands[::3],
              colors="white", linewidths=0.6)
raw_walk = path(temperature, 0.0015, 400)
right.plot(raw_walk[:, 0], raw_walk[:, 1], "o-", color="#0072B2", markersize=2.5, linewidth=1.2)
right.plot([exact_slope], [exact_intercept], "*", color="#000000", markersize=16)
right.annotate("after 400 steps,\nstill here", xy=(raw_walk[-1, 0], raw_walk[-1, 1]),
               xytext=(16.9, 9), arrowprops=dict(arrowstyle="-|>", color="#000000"), fontsize=9)
right.set_xlim(15.2, 18.2)
right.set_ylim(-2, 31)
right.set_title("raw degrees: stripes, and a crawl")
right.set_xlabel("slope")
right.set_ylabel("intercept")

plt.tight_layout()
plt.show()
print("raw run, slope     : %.4f -> %.4f   (target %.4f)"
      % (raw_walk[1, 0], raw_walk[-1, 0], exact_slope))
print("raw run, intercept : %.4f -> %.4f   (target %.4f)"
      % (raw_walk[1, 1], raw_walk[-1, 1], exact_intercept))

On the right the contours are not circles, they are **near-parallel stripes** - a valley so long that
its ends are off the edge of the picture. The walk drops onto the valley floor within a couple of steps,
getting the slope roughly right almost immediately, and then has nowhere to go but along the floor,
where the ground barely tilts. After 400 steps the intercept has moved from 1.00 to 3.13 and needs to
reach 27.27.

**That is the whole cost of not scaling, in one image.** The direction of steepest descent is only a
good direction to walk when the contours are round. When they are stripes, "steepest" points across the
valley rather than along it, and almost every step is spent going back and forth rather than forward.

### 115 steps against 21,769

The standardised version converges in **115 steps**. The raw version needs **21,769** at the fastest of
the rates that still work, and 58,236 at a more cautious one - and at 0.01 it is already diverging, so
there is not much room above 0.0015. That is a factor of nearly two hundred, for a change that touches
no data and alters no answer.

**Why.** With raw temperatures the two parameters have wildly different sensitivities: a change of
0.001 in the slope moves every prediction by up to 0.03, while the same change in the intercept moves
them by 0.001. The loss surface is not a round bowl but a long narrow valley, and gradient descent
zigzags across it, making progress along the valley floor only slowly. The learning rate must be small
enough for the *steep* direction, which makes it far too small for the shallow one.

There is a single number that measures this, and it is worth meeting because it explains several
things at once.

In [ ]:
for label, feature in [("raw degrees ", temperature), ("standardised", scaled_temp)]:
    design = np.column_stack([np.ones(n_days), feature])
    print("%s  condition number %8.1f" % (label, np.linalg.cond(design.T @ design)))

**5482.0 against 1.0.** The condition number is the ratio between the steepest and shallowest
directions of the bowl - it is 1 for a perfectly round one. Standardising turned a valley 5,482 times
longer than it is wide into a circle.

**This is why scaling matters even for methods that have nothing to do with distance.** 03-07 needed
it because distance is measured in the largest unit; here it is needed because the optimiser's step
size has to suit every direction at once. The two reasons are unrelated and the fix is the same, which
is why "standardise your features" is close to universal advice - and why it is worth knowing that
trees, which neither measure distance nor descend gradients, are the exception.

Real datasets have features on far more disparate scales than 10-to-30 degrees. A model mixing incomes
and ages, unscaled, has a condition number in the millions, and the practical symptom is a fit that
never seems to converge no matter how long it runs.

## Common misconceptions

**"The loss is the same thing as the metric I report."**
It is not, and the difference matters. The loss is what the optimiser can walk downhill on; the metric
is what the decision is judged by. Squared error is chosen partly because it is smooth, not because
anybody cares about squared euros. Module 06 is entirely about this gap.

**"Gradient descent finds the best answer."**
It finds a point where the ground is flat. For a squared-error linear model that is the one true bottom,
because the bowl has only one. For a neural network there are many, and which one you land in depends on
where you started.

**"A smaller learning rate is safer."**
Safer against divergence, and a reliable way to never arrive. 0.001 in this chapter was moving in exactly
the right direction and still sat at a loss of 24,582 against a possible 431.78 after 400 steps.

**"The loss going down means the model is improving."**
It means the fit to *this* data is improving. Push it far enough and the model starts learning the noise -
that is chapter 05-01, and it is why training loss alone can never tell you when to stop.

**"You need calculus to understand this."**
The derivative was computed in this chapter with two evaluations and a subtraction, and it matched the
calculus to four decimals. Calculus makes it exact and fast; it is not what makes it work.

**"Scaling is for distance-based methods."**
It is for anything that takes steps. The condition number here went from 5,482 to 1, and the step count
from 21,769 to 115, in a method that never measures a distance between rows.

**"Divergence means there is a bug."**
Usually it means the step is too long. Each step still points downhill - it just lands further up the
opposite wall. Divide the rate by ten before you look for a bug.

## Exercises

Solutions: `solutions/03_math_foundations/03-08_loss_and_gradients_solutions.ipynb`.

Most of these are arithmetic on purpose. Gradient descent is one of the few ideas in the course you can
fully verify by hand, and doing so once removes the mystery permanently.

### Quick understanding

**E1.** A loss function is given 60 rows of data and a candidate slope and intercept. How many numbers
does it return, and what does "lower is better" mean about the sign of the residuals?

**E2.** Grid search over 100 values per parameter takes 10,000 evaluations for two parameters. Write down
the count for six parameters, and say in one sentence what information gradient descent uses that grid
search throws away.

**E3.** The gradient at your current position is `(-4185, -193)`. Which way do you move each parameter,
and which of the two will move further for the same learning rate?

**E4.** Your loss curve is still visibly falling at the last step. Give the two possible causes and the
fix for each.

### Hand calculation

Use this three-row dataset for E5 to E8. Do the arithmetic on paper first; the workspace cell is there
to check yourself afterwards.

| row | `x` | `y` |
|---|---|---|
| 1 | 1 | 3 |
| 2 | 2 | 5 |
| 3 | 3 | 7 |

The model is `prediction = intercept + slope x`, the loss is the mean squared error, and the gradients
are `2 x mean(residual x x)` and `2 x mean(residual)` with `residual = prediction - y`.

**E5.** Starting from slope 0 and intercept 0, compute the loss, then both gradients. Show the three
residuals on the way.

**E6.** Take one step with a learning rate of 0.1. Report the new slope, the new intercept and the new
loss. By what factor did the loss fall?

**E7.** Take a second step from there. Report the new slope, intercept and loss, and say what has
happened to the *size* of the gradients compared with E5.

**E8.** Now redo E6 with a learning rate of 1.0 instead of 0.1. Report the new loss and explain, using
the picture from the failure lab, why one step was enough to ruin it.

**E9.** The exact fit for that data is `y = 1 + 2x`, with a loss of zero. Two steps at 0.1 got you close
but not exact. Explain why gradient descent never *quite* arrives, and what practitioners do about it.

**E10.** Estimate the derivative of `f(x) = x^2` at `x = 3` two ways with `h = 0.5`: the central
difference `(f(3+h) - f(3-h)) / 2h` and the forward difference `(f(3+h) - f(3)) / h`. The true answer is
6. One of them is exactly right even with a step this large - say which, and why that is not luck.

**E11.** Repeat E10 for `f(x) = x^3` at `x = 2`, where the true derivative is 12. Both methods are now
wrong. Which is closer, and what does this tell you about which one to use when checking a gradient?

**E12.** In the chapter, the derivative of the loss with respect to the slope was -12,681.61 at slope 0,
and it changed by exactly 849.61 for every unit of slope. Without evaluating the loss again, compute
where the derivative reaches zero. Then say what the equivalent calculation costs for a model with a
million parameters, and why that rules it out.

**E13.** On standardised data each step multiplies the remaining distance to the answer by
`1 - 2 x learning_rate`. The distance after step 1 was 311.53. Using a learning rate of 0.05, compute by
hand how many further steps are needed to bring it below 0.01. (`log(0.01/311.53) / log(0.9)`.)

**E14.** Using the same rule, which learning rate converges fastest, and what happens at exactly that
value? For which learning rates does the method diverge? State the range as an inequality.

**E15.** Five runs on the standardised data produced these losses. The best achievable is 431.78.
Label each run *too small*, *well chosen* or *diverging*, and rank the two workable ones.

| run | step 1 | step 10 | step 50 |
|---|---|---|---|
| A | 120,004.9 | 117,870.8 | 108,837.3 |
| B | 117,860.1 | 98,427.4 | 44,287.1 |
| C | 97,480.0 | 14,998.2 | 435.0 |
| D | 431.8 | 431.8 | 431.8 |
| E | 145,405.0 | 806,471.2 | 1,651,091,649.7 |

### Coding

**E16.** Add early stopping to `fit`: return as soon as the loss improves by less than a tolerance you
pass in. Run it at 0.05 with a tolerance of 1e-9 and report the step it stopped at and the loss there.

**E17.** Generalise `fit` to any number of features using `X @ w` from 03-07. Test it on three features
built from `temperature` (the temperature, its square, and a random column) and check the weights against
`np.linalg.lstsq`.

**E18.** Write `check_gradient(function, analytic, at)` comparing a supplied analytic gradient with the
numerical one and returning the relative difference. Then deliberately break the analytic gradient - drop
the factor of 2 - and confirm your checker catches it. This is the standard test when implementing a
model.

**E19.** Change the loss from squared error to **absolute** error and descend on that instead. (Its
gradient with respect to the slope is `mean(sign(residual) x feature)`.) Fit both on the chapter's data,
report both lines, and say which rows account for the difference.

**E20.** Empirically find the largest learning rate that still converges on the standardised data, to two
decimal places. Compare it with `1 / mean(scaled_temp ** 2)` and explain the relationship.

### Interpretation

**E21.** A colleague reports that their training loss has fallen for 200 epochs straight and concludes
the model is good. Name the check they have not done and the failure it would catch.

**E22.** A model mixing `age` (20-70) and `income` (20,000-90,000) has a condition number around 8e+10
unscaled. Describe the two symptoms the person fitting it will actually observe, before anyone mentions
condition numbers.

### Debugging

**E23.** Someone writes `slope += learning_rate * gradient_slope` instead of `-=`. Describe precisely
what the loss curve will look like, and how you would confirm the diagnosis in one plot.

**E24.** A fit produces `nan` after nine steps. List, in the order you would try them, the three things
to check.

### Exam and interview reasoning

**E25.** Explain gradient descent in 60 seconds to an interviewer, without a whiteboard. Then answer the
follow-up: "if Newton's method converges in one step, why does nobody use it for deep networks?"

### Transfer to a different situation

**E26.** You are fitting a model that predicts a probability, so squared error is the wrong loss. Nothing
about the five-line loop changes except one thing. Say what changes, and what stays exactly the same.

### Explain it to someone non-technical

**E27.** Explain in under 90 words how a computer "learns" a line from data, using neither the word
gradient nor the word derivative.

### Optional challenge

**E28.** Add **momentum** to `fit`: keep a running velocity `v = 0.9 * v + gradient` and step along `v`
instead of the gradient. Run it on the *unscaled* feature and compare the step count with the 21,769 of
the chapter. Momentum is designed for exactly the narrow-valley problem, so this is a fair test of
whether you have understood what the valley was.

In [ ]:
# Your workspace. In memory: temperature, rentals, scaled_temp, mean_temp, sd_temp,
# loss, derivative, fit, path, steps_to_converge, walk, history,
# exact_slope, exact_intercept, best_slope, best_intercept.

## Mastery check

- [ ] Write down a loss function and say what one evaluation of it returns
- [ ] Give the number that kills grid search, and say what gradient descent uses instead
- [ ] Compute a derivative numerically with two evaluations, and check it against an analytic one
- [ ] Read the sign *and the size* of a gradient, and say what each tells you
- [ ] Write the five-line loop from memory
- [ ] Do two steps of gradient descent by hand on three rows and get the right numbers
- [ ] Diagnose too-small, well-chosen and diverging learning rates from a loss curve alone
- [ ] Explain, with the contour picture, why unscaled features cost a factor of two hundred
- [ ] State what a condition number of 1 means and why standardising achieves it

## What should now feel instinctive

- Plotting the loss curve first whenever a fit misbehaves, before changing anything else
- Reaching for "divide the learning rate by ten" as the first response to a `nan`
- Standardising features before fitting anything that takes steps, not only anything that measures
  distance
- Checking an analytic gradient against a numerical one the first time you write it
- Reading "the loss went down" as a statement about the training data and nothing else

## Flashcards

| Front | Back |
|---|---|
| Loss function | One number saying how badly a candidate answer fits. Lower is better |
| Mean squared error | Average of the squared residuals. Smooth, and heavily punishes large errors |
| Why grid search fails | 100 values per parameter is 1e+20 evaluations at 10 parameters |
| Derivative, numerically | `(f(x+h) - f(x-h)) / 2h`. Two evaluations, no calculus |
| Gradient | The vector of one derivative per parameter. Points uphill; its negative points downhill |
| Gradient of MSE | `2 x mean(residual x feature)` for a weight, `2 x mean(residual)` for the intercept |
| Gradient descent | Start anywhere, step against the gradient, repeat |
| Learning rate | The step size. The only setting, and a cheap stand-in for `1 / curvature` |
| Loss still falling at the end | Rate too small or too few steps |
| Loss rising or `nan` | Rate too large. Divide by ten |
| Why steps shrink near the bottom | The gradient shrinks there. Nobody programmed it |
| Condition number | Ratio of the steepest to the shallowest direction. 1 is a round bowl |
| Cost of not scaling here | 5,482 instead of 1, and 21,769 steps instead of 115 |
| Newton's method | Divide by the curvature instead of guessing. One step, but `p x p` work |
| Backpropagation | The chain rule computing these gradients automatically, layer by layer |

## Next

**Module 04 · The machine learning workflow**, starting with **04-01 · Framing a problem**.

Module 03 is finished, and with it the mathematics. You can summarise a column, say how much to trust a
number computed from a sample, update a belief with evidence, read a slope with its units, write a
prediction as `X @ w`, and now choose the `w` by defining what wrong means and walking downhill.

What none of that tells you is *which question to ask*. The next module is about the part that surrounds
the model: turning a vague business request into a target with a unit, choosing an honest way to split
the data, building the baseline the model has to beat, and knowing what "good" would mean before you fit
anything. It is the part that decides whether the modelling was worth doing, and the part that beginners
skip.